In [1]:
import zarr
from matplotlib import pyplot as plt
import numpy as np
import dask.array as da
import tensorflow as tf
#from torch import bfloat16

bfloat16 = tf.bfloat16.as_numpy_dtype

2023-07-25 17:31:13.446354: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:

aia_path = "/mnt/sdomlv2_full/sdomlv2.zarr"
aia_data = zarr.group(zarr.DirectoryStore(aia_path))

HALF_PRECISION = True
USE_BFLOAT = False

In [3]:
def copy_zarr_group(source_group, target_group):
    try:
        # Copy arrays (datasets) from the source group to the target group
        for array_name, zarr_array in source_group.items():
            shape = zarr_array.shape
            #dtype = zarr_array.dtype
            dtype = 'bfloat16' if USE_BFLOAT else zarr_array.dtype
            target_group.create_dataset(array_name, shape=shape, dtype=dtype)
    except:

        # Recursively copy nested groups
        for group_name, nested_group in source_group.groups():
            new_nested_group = target_group.create_group(group_name)
            copy_zarr_group(nested_group, new_nested_group)

In [4]:
output_path = '/home/willfaw/data' if not HALF_PRECISION else '/home/willfaw/data16'
if USE_BFLOAT: 
    output_path += 'bfloat'
print(output_path)

new_group = zarr.group(store=output_path)

copy_zarr_group(aia_data, new_group)

new_group.info

/home/willfaw/data16


Name,/
Type,zarr.hierarchy.Group
Read-only,False
Store type,zarr.storage.DirectoryStore
No. members,11
No. arrays,0
No. groups,11
Groups,"2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020"


In [5]:
years = ['2010', '2011', '2012', '2013', '2014'] #, '2015', '2016', '2017', '2018', '2019', '2020']
wavelengths = ['131A', '1600A', '1700A', '171A', '193A', '211A', '304A', '335A', '94A']

#years = ['2010']
wavelengths = ['131A']
#wavelengths = ['1600A', '1700A', '171A', '193A', '211A', '304A', '335A', '94A']


In [6]:
# Hack: randomly sub-sample the aia data to speed up normalization calculation

def print_array_info(array):
        print(f"\tmean: {array.mean()}, max: {array.max()}, min: {array.min()}")
        print(f"\tarray size: {array.size}, memory size of one element: {array.itemsize}, total memory: {array.size * array.itemsize}")



print("activate hack")
fraction_size = 0.01
#years = ['2010']

for year in years: 
    for wavelength in wavelengths:
        print(year, wavelength)
        total_elements = aia_data[year][wavelength].shape[0]
        num_elements_to_select = int(total_elements * fraction_size)
        random_indices = np.random.choice(total_elements, num_elements_to_select, replace=False)
        print(f"Before: {aia_data[year][wavelength].shape[0]}, selcected {num_elements_to_select} elements. Writing ...")
        #new_array = aia_data[year][wavelength].get_coordinate_selection(random_indices)
        #new_array = da.from_array(aia_data[year][wavelength])[random_indices]
        new_array = aia_data[year][wavelength][random_indices]

        print_array_info(new_array)
        if HALF_PRECISION:
            new_array /= 2 
            if USE_BFLOAT:
                new_array = new_array.astype(bfloat16)
            else:
                new_array = new_array.astype(np.float16)
            print_array_info(new_array)
        new_group[year][wavelength] = new_array
        print(f"After: {new_group[year][wavelength].shape[0]}")
        print('-'*20)

activate hack
2010 131A
Before: 47116, selcected 471 elements. Writing ...


	mean: 3.2640626430511475, max: 3832.318115234375, min: 0.0
	array size: 123469824, memory size of one element: 4, total memory: 493879296
	mean: 1.6318359375, max: 1916.0, min: 0.0
	array size: 123469824, memory size of one element: 2, total memory: 246939648
After: 471
--------------------
2011 131A
Before: 75200, selcected 752 elements. Writing ...
	mean: 4.2603559494018555, max: 6372.10107421875, min: 0.0
	array size: 197132288, memory size of one element: 4, total memory: 788529152
	mean: 2.130859375, max: 3186.0, min: 0.0
	array size: 197132288, memory size of one element: 2, total memory: 394264576
After: 752
--------------------
2012 131A
Before: 76849, selcected 768 elements. Writing ...
	mean: 4.443510055541992, max: 76617.8046875, min: 0.0
	array size: 201326592, memory size of one element: 4, total memory: 805306368
	mean: 2.22265625, max: 38304.0, min: 0.0
	array size: 201326592, memory size of one element: 2, total memory: 402653184
After: 768
--------------------
2013 13

/var/tmp/ipykernel_1136693/3974975214.py:30: RuntimeWarning: overflow encountered in cast
  new_array = new_array.astype(np.float16)


	mean: inf, max: inf, min: 0.0
	array size: 216793088, memory size of one element: 2, total memory: 433586176
After: 827
--------------------
2014 131A
Before: 73605, selcected 736 elements. Writing ...
	mean: 5.171034336090088, max: 62844.640625, min: 0.0
	array size: 192937984, memory size of one element: 4, total memory: 771751936
	mean: 2.5859375, max: 31424.0, min: 0.0
	array size: 192937984, memory size of one element: 2, total memory: 385875968
After: 736
--------------------
